# Job Task: Deployment Gate & Register

> **This notebook runs as a Databricks Job task.**

Task 3 of the retrain pipeline:
- Load the challenger model (from Task 2) and the current champion
- Run the deployment gate: fail if F1 drops by more than 5%
- If gate passes: register the challenger as the new champion
- If gate fails: the job stops here (serving endpoint is NOT updated)

In [0]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
dbutils.widgets.text("schema", "00_shared", "Schema")
dbutils.widgets.text("config_path", "config.yml", "Config Path")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

In [0]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml, mlflow
from mlflow import MlflowClient

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
config_path  = dbutils.widgets.get("config_path")

with open(config_path) as f:
    config = yaml.safe_load(f)

# Get challenger run_id from previous task
best_run_id = dbutils.jobs.taskValues.get(taskKey="train_models", key="best_run_id")
print(f"Challenger run ID: {best_run_id}")

# Define model name for the shared workshop model
model_name = f"{catalog}.{schema}.churn_classifier"
champion_uri = f"models:/{model_name}@champion"

# Run deployment gate
from churn_model.evaluate import evaluate_gate
test_df = spark.table(f"{catalog}.`00_shared`.telco_churn_holdout").toPandas()

result = evaluate_gate(
    challenger_run_id=best_run_id,
    champion_model_uri=champion_uri,
    test_data=test_df,
    config=config,
    threshold=0.05,
)

print(result.reason)

if not result.should_deploy:
    raise ValueError(f"Deployment gate failed: {result.reason}")

# Gate passed — register the challenger as the new version
mlflow.set_registry_uri("databricks-uc")
new_version = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name=model_name,
)

client = MlflowClient()
client.set_registered_model_alias(model_name, "champion", new_version.version)

print(f"✓ New champion registered: version {new_version.version}")
dbutils.jobs.taskValues.set("new_version", str(new_version.version))